In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./config/NaBr.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

In [3]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3


trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

config['model_input_fields'] = {'node_spin': o3.Irreps('1x1e')}

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

Torch device: cpu
/Users/temporary/Documents/GitHub/pytorch-intel-mps/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


Is ij diagonal
tensor(True)


Replace string dataset_forces_rms to 0.3425767123699188
Replace string dataset_per_atom_total_energy_mean to -2.96089243888855
Atomic outputs are scaled by: [Na, Br: 0.342577], shifted by [Na, Br: -2.960892].
Replace string dataset_forces_rms to 0.3425767123699188
Initially outputs are globally scaled by: 0.3425767123699188, total_energy are globally shifted by None.


In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
    
from torch import nn
import math

data0 = AtomicData.to_AtomicDataDict(dataset[0])
#data0[AtomicDataDict.NODE_SPIN] = torch.randn_like(data0['pos'], device='cuda')

#data1 = with_edge_spin_length(data0, with_distance = True)

In [6]:
trainer.model = final_model

In [7]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math

data_new = final_model(data0)

In [8]:
trainer.

Number of weights: 37416
Number of trainable weights: 37416
! Starting training ...

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      0     6        0.888         0.78        0.108        0.214        0.303         7.17        0.112


  Initialization     #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Initial Validation          0    1.004    0.002        0.868        0.127        0.996        0.226        0.319         7.72        0.121
Wall time: 1.0049701030366123
! Best model        0    0.996

training
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      1    10        0.643        0.643     1.68e-05        0.206        0.275       0.0899       0.0014
      1    20        0.656         0.64       0.0164        0.201        0.274         2.81       0.0439
      1    30        0.2

      4    40       0.0495       0.0493     0.000171       0.0571       0.0761        0.287      0.00448
      4    50        0.058       0.0544      0.00361        0.063       0.0799         1.32       0.0206
      4    60       0.0301         0.03     6.45e-05        0.047       0.0593        0.176      0.00275
      4    70       0.0349       0.0322      0.00266       0.0495       0.0615         1.13       0.0177
      4    80       0.0368       0.0367      0.00012       0.0537       0.0656         0.24      0.00375
      4    90       0.0541       0.0474      0.00676       0.0593       0.0745          1.8       0.0282
      4   100       0.0711        0.071     6.99e-06       0.0643       0.0913        0.058     0.000906
      4   110       0.0247       0.0242     0.000557       0.0421       0.0533        0.517      0.00809
      4   120       0.0301       0.0262      0.00389       0.0441       0.0555         1.37       0.0214
      4   130       0.0465       0.0463     0.000251   

In [10]:
!nequip-evaluate --train-dir results/NaBr-tutorial/NaBr --batch-size 1

Traceback (most recent call last):
  File "/Users/temporary/anaconda3/bin/nequip-evaluate", line 5, in <module>
    from nequip.scripts.evaluate import main
  File "/Users/temporary/anaconda3/lib/python3.9/site-packages/nequip/scripts/evaluate.py", line 12, in <module>
    import torch
  File "/Users/temporary/anaconda3/lib/python3.9/site-packages/torch/__init__.py", line 229, in <module>
    from torch._C import *  # noqa: F403
ImportError: dlopen(/Users/temporary/anaconda3/lib/python3.9/site-packages/torch/_C.cpython-39-darwin.so, 0x0002): Library not loaded: @loader_path/libtorch_cpu.dylib
  Referenced from: <9F28170E-1091-3C0E-B17E-71871B0E6A44> /Users/temporary/anaconda3/lib/python3.9/site-packages/torch/lib/libtorch_python.dylib
  Reason: tried: '/Users/temporary/anaconda3/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib' (no such file), '/usr/local/lib/libtorch_cpu.dylib' (no such file), '/usr/lib/libtorch_cpu.dylib' (no such file, not in dyld cache)
